In [1]:
# 1. Импорты
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix

# 2. Загрузка данных
df = pd.read_csv('../data/train.csv')

# 3. Отбор признаков
# Создаём признак "IsAlone" (одиночка ли пассажир)
df['IsAlone'] = (df['SibSp'] + df['Parch'] == 0).astype(int)

# Оставляем только нужные колонки
features = ['Age', 'Pclass', 'Sex', 'IsAlone']
X = df[features]
y = df['Survived']

print("Первые 5 строк отобранных признаков:")
print(X.head())

# 4. Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Предобработка (заполнение пропусков, кодирование категорий, масштабирование)
# Копируем, чтобы не менять исходный DataFrame
X_train_processed = X_train.copy()
X_test_processed = X_test.copy()

# 5a. Заполняем пропуски в Age медианой (обучаем на train, применяем к test)
age_median = X_train_processed['Age'].median()
X_train_processed['Age'] = X_train_processed['Age'].fillna(age_median)
X_test_processed['Age'] = X_test_processed['Age'].fillna(age_median)

# 5b. Кодируем Sex: male -> 0, female -> 1
X_train_processed['Sex'] = X_train_processed['Sex'].map({'male': 0, 'female': 1})
X_test_processed['Sex'] = X_test_processed['Sex'].map({'male': 0, 'female': 1})

# 5c. Масштабируем числовые признаки (Age, Pclass, IsAlone - хотя Pclass и так в порядке)
scaler = StandardScaler()
numeric_cols = ['Age', 'Pclass', 'IsAlone']
# Обучаем scaler на train, преобразуем train и test
X_train_processed[numeric_cols] = scaler.fit_transform(X_train_processed[numeric_cols])
X_test_processed[numeric_cols] = scaler.transform(X_test_processed[numeric_cols])

# 6. Обучение модели
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_processed, y_train)

# 7. Предсказание и оценка
y_pred = model.predict(X_test_processed)
y_pred_proba = model.predict_proba(X_test_processed)[:, 1]

print("\n=== Результаты на тестовой выборке ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 8. Коэффициенты модели (важность признаков)
coef_df = pd.DataFrame({
    'feature': features,
    'coef': model.coef_[0]
}).sort_values('coef', ascending=False)

print("\n=== Важность признаков (коэффициенты) ===")
print(coef_df)

Первые 5 строк отобранных признаков:
    Age  Pclass     Sex  IsAlone
0  22.0       3    male        0
1  38.0       1  female        0
2  26.0       3  female        1
3  35.0       1  female        0
4  35.0       3    male        1

=== Результаты на тестовой выборке ===
Accuracy: 0.7821
ROC-AUC:  0.8355

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.84      0.83       110
           1       0.73      0.70      0.71        69

    accuracy                           0.78       179
   macro avg       0.77      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179


Confusion Matrix:
[[92 18]
 [21 48]]

=== Важность признаков (коэффициенты) ===
   feature      coef
2      Sex  2.530224
3  IsAlone  0.012673
0      Age -0.445512
1   Pclass -0.969262
